# Sparse radio map reconstruction on a Sionna RT street canyon

Classical interpolation baselines scored two ways: filling scattered gaps, and
predicting into a block of street that was never measured. The two answers differ
by a factor of five.

Runs top to bottom on a **free Colab CPU runtime**. No GPU required.

Repo: https://github.com/alioramuss/radiomap-sparse-reconstruction

---

**Two paths through this notebook.**

1. *With Sionna* (cell 2 below): traces the real ground-truth map. The 2e8-ray map
   takes about 86 s on two CPU cores; installing Sionna on Colab takes a few minutes.
2. *Without Sionna*: uses a piecewise-smooth stand-in field so the whole pipeline runs
   in under a minute. Useful for reading the code, **not** for producing results — no
   number in the report comes from it.


## 0. Setup


In [ ]:
!pip -q install numpy scipy matplotlib
!git clone -q https://github.com/alioramuss/radiomap-sparse-reconstruction.git 2>/dev/null || true
%cd radiomap-sparse-reconstruction

import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=2, suppress=True)


In [ ]:
# Set to True to trace the real map. Needs Sionna and a few minutes to install.
USE_SIONNA = False

if USE_SIONNA:
    !pip -q install sionna


## 1. The ground-truth map

Sionna RT 2.0.1, the built-in `simple_street_canyon`, one isotropic rooftop
transmitter at 30 dBm, 3.5 GHz, 1 m cells at a 1.5 m measurement plane.

Two things the docs do not say:

- `path_gain` is a **linear** power ratio, not dB. Values here are of order 1e-9.
- `refraction` defaults to **on**. Left on, rays leak into building interiors and
  those cells stop being structurally unreachable, which quietly changes what an
  accessibility mask means.


In [ ]:
from radiomap.config import SceneConfig, SweepConfig
from radiomap.scene import synthetic_canyon, trace

if USE_SIONNA:
    rm = trace(SceneConfig())
else:
    rm = synthetic_canyon()

n_cells = rm.grid_shape[0] * rm.grid_shape[1]
print(f'grid {rm.grid_shape}, {n_cells} cells')
print(f'{rm.building_mask.sum()} under buildings, {rm.n_valid} valid ({100 * rm.n_valid / n_cells:.1f}%)')
print(f'gain {rm.gain_db.min():.0f} to {rm.gain_db.max():.0f} dB, sd {rm.gain_db.std():.1f} dB')


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
im = ax.imshow(rm.to_grid(rm.gain_db), origin='lower', cmap='viridis')
ax.set_title('Path gain (dB). White cells are under buildings.')
ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(im, ax=ax, label='dB', fraction=0.03)
plt.show()


## 2. How much of the map is simulator noise?

Before comparing reconstruction errors of a few dB, it is worth asking how
repeatable the ground truth is. Tracing the same scene twice with different random
seeds gives two maps that should be identical and are not.

At **2e5 rays** — the count used by the closest published Sionna RT reconstruction
study — 29% of open-street cells receive no ray at all, and the rest disagree
between seeds by **4.75 dB**. Reconstruction errors in this literature are typically
2–5 dB. At that ray count the simulator disagrees with itself by more than the
effect being measured.

This notebook runs at 2e8, where a single map sits **0.23 dB** from the converged
mean. That costs 86 s.

Run `scripts/02_ray_convergence.py` for the full sweep (it needs Sionna).


## 3. The variogram

Fitted on a 10% draw, which is what a real user would have, not on the full map.
The correlation range sets the hold-out block size: a block narrower than the
correlation range is still measuring interpolation, not extrapolation.

A near-zero nugget is also a second, independent check that the ground truth is not
noisy — independent per-cell noise would appear as variance that does not vanish at
zero lag.


In [ ]:
from radiomap.variogram import empirical_variogram, fit_spherical

cfg = SweepConfig()
rng = np.random.default_rng(cfg.variogram_fit_seed)
idx = rng.choice(rm.n_valid, size=int(cfg.variogram_fit_fraction * rm.n_valid), replace=False)

lags, gamma, counts = empirical_variogram(rm.xy[idx], rm.gain_db[idx])
model = fit_spherical(lags, gamma, counts)
print(model)
print(f'nugget is {100 * model.nugget / model.total_sill:.1f}% of the total sill')
print(f'block side {cfg.block_side_m:.0f} m is {cfg.block_side_m / model.rng:.1f}x the correlation range')

plt.figure(figsize=(6, 3.5))
plt.plot(lags, gamma, 'o', ms=4, label='empirical')
plt.plot(lags, model(lags), '-', label='spherical fit')
plt.xlabel('lag (m)'); plt.ylabel('semivariance (dB^2)')
plt.legend(frameon=False); plt.grid(alpha=0.25); plt.show()


## 4. The two sampling designs

**Random** draws observations uniformly from the valid cells and scores on the rest.
Every unobserved cell has an observation nearby, so this measures gap filling. It is
what nearly all of this literature does.

**Block** removes a contiguous 60 m square entirely, draws observations from what
remains, and scores only inside the square. Nothing inside is ever observed, so it
measures extrapolation into unseen ground.

Errors are scored on unobserved cells only. Including observed cells would flatter
IDW, RBF and kriging, all of which reproduce their own observations exactly, and the
flattery would grow with sampling density.


In [ ]:
from radiomap.sampling import block_design, block_origins, random_design

obs_r, sc_r = random_design(rm.n_valid, 0.05, seed=0)
origins = block_origins(rm.xy, cfg.block_side_m, cfg.n_block_positions)
obs_b, sc_b = block_design(rm.xy, origins[0], cfg.block_side_m, 0.05, seed=0)

print(f'random: {len(obs_r)} observed, {len(sc_r)} scored')
print(f'block : {len(obs_b)} observed, {len(sc_b)} scored, {len(origins)} block positions available')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (o, s, t) in zip(axes, [(obs_r, sc_r, 'Random'), (obs_b, sc_b, 'Block')]):
    ax.scatter(rm.xy[s, 0], rm.xy[s, 1], s=1, c='#d8dee5', label='scored')
    ax.scatter(rm.xy[o, 0], rm.xy[o, 1], s=2, c='#1c5aa8', label='observed')
    ax.set_title(f'{t} design, 5%'); ax.set_aspect('equal')
    ax.legend(markerscale=4, frameon=False, fontsize=8)
plt.show()


## 5. One split, all five methods

Hyperparameters are chosen by 5-fold cross-validation on the observations only,
never against the held-out truth. With the hold-out design the tempting shortcut is
to tune against the block, which turns an extrapolation experiment into an
interpolation one with extra steps.


In [ ]:
from radiomap.sweep import run_once

for design, (o, s) in [('random', (obs_r, sc_r)), ('block', (obs_b, sc_b))]:
    print(f'--- {design} ---')
    for method in cfg.methods:
        row = run_once(rm, method, o, s, cfg)
        print(f"  {method:<8} RMSE {row['rmse']:6.2f} dB   MAE {row['mae']:6.2f} dB   {row['params']}")


## 6. Inside one hold-out block

This is where the number stops being abstract. The truth inside the block is not a
smooth field: it is piecewise smooth, cut by sharp edges where a diffraction wedge
or a specular reflection boundary starts and stops.

IDW, RBF and kriging all assume the field varies smoothly with distance. Under
random sampling that assumption survives, because observations land on both sides of
every edge. Under hold-out there is nothing inside the block to pin anything down,
so every method returns a smooth surface — and the error map is the edges themselves.


In [ ]:
from radiomap.crossval import select
from radiomap.interpolators import METHODS
from radiomap.metrics import rmse

extra = {'tx_xy': rm.tx_xy}
params, _ = select('dk', rm.xy[obs_b], rm.gain_db[obs_b], folds=cfg.cv_folds,
                   seed=cfg.cv_seed, extra=extra)
model = METHODS['dk'](**{**params, **extra}).fit(rm.xy[obs_b], rm.gain_db[obs_b])
pred = model.predict(rm.xy[sc_b])
truth = rm.gain_db[sc_b]

xy = rm.xy[sc_b]
xs, ys = np.unique(xy[:, 0]), np.unique(xy[:, 1])
ix, iy = np.searchsorted(xs, xy[:, 0]), np.searchsorted(ys, xy[:, 1])

def grid(v):
    g = np.full((len(ys), len(xs)), np.nan); g[iy, ix] = v; return g

lim = float(np.nanpercentile(np.abs(pred - truth), 98))
lo, hi = np.nanpercentile(truth, [2, 98])

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (g, t, kw) in zip(axes, [
    (grid(truth), 'Truth', dict(cmap='viridis', vmin=lo, vmax=hi)),
    (grid(pred), 'Prediction', dict(cmap='viridis', vmin=lo, vmax=hi)),
    (grid(pred - truth), 'Error', dict(cmap='RdBu_r', vmin=-lim, vmax=lim)),
]):
    im = ax.imshow(g, origin='lower', **kw)
    ax.set_title(t); ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(im, ax=ax, fraction=0.046, label='dB')
fig.suptitle(f'One hold-out block at 5% sampling, RMSE {rmse(truth, pred):.2f} dB')
plt.tight_layout(); plt.show()


## 7. The full sweep

Five fractions x five methods x (5 random seeds | 6 block positions x 3 seeds).
That is 575 fits (125 random, 450 block) and it takes roughly an hour on two cores, so it is left as a
script rather than run inline here:

```bash
python scripts/03_reconstruction_sweep.py --map data/map_3p5GHz.npz --out results/sweep.csv
python scripts/05_figures.py --map data/map_3p5GHz.npz --sweep results/sweep.csv
```

The headline from the full run at 3.5 GHz:

| | 1% | 20% |
|---|---|---|
| best under random sampling | 5.60 dB | **2.08 dB** |
| best under 60 m hold-out | 12.01 dB | **9.68 dB** |

Twenty times more data buys 63% under random sampling and 2.3 dB under hold-out,
and the hold-out curve is flat from 5% onward. Samples outside a gap do not help you
fill it once you have enough of them to pin down the surroundings.

The missing information is not *what is the level around here* — the surroundings
already answer that. It is *where does the shadow boundary run*, which is a fact
about geometry that no amount of sampling elsewhere in the street recovers.


## 8. Checks

Nineteen known-answer checks. Each encodes a fact that has to be true for the
numbers above to mean what they say.


In [ ]:
!python tests_pipeline.py
